# PyDI Data Integration Workflow: Companies

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with companies datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [ ]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [138]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [139]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")
spec.set_column("founders", output_type="list<string>")
target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,website,string
3,founded,datetime
4,country,string
5,city,string
6,industry,string
7,assets,int
8,revenue,int
9,founders,list<string>


## Step 2: Load Source Datasets

In [170]:
from PyDI.io import load_csv, load_xml, load_json
import numpy as np
dbpedia = load_json(INPUT_DIR / "schemamatching" / "original_data" / "dbpedia.json")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia["founders"] = dbpedia["founders"].apply(lambda x: x if x else np.NaN)
dbpedia.head(10)

,identifier,name,countryName,fyear,cityName,revenue,assets,industryName,homepage,founders
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,France,1970-01-01,Castelnaudary,,,Meat,,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,ÿguas de Portugal,Portugal,1993-01-01,Lisbon,,,,,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,ÿnima Estudios,Mexico,2002-01-01,Mexico City,,,Animation,,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalik Enerji,Turkey,1998-01-01,Istanbul,,,,,[Çalik Holding]
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalik Holding,Turkey,1997-01-01,Istanbul,2.80,8.00,,,[Ahmet Çalik]
5,http://dbpedia.org/resource/%C3%87ukurova_(con...,Çukurova (construction firm),Turkey,1975-01-01,Istanbul,,,,,NaN
6,http://dbpedia.org/resource/%C3%87ukurova_Holding,Çukurova Holding,Turkey,1923-01-01,Istanbul,,,Communication,,NaN
7,http://dbpedia.org/resource/%C3%89lectricit%C3...,Électricité de France,France,1946-01-01,Paris,"65,170,000,000.00","240,560,000,000.00",Electric utility,,[Marcel Paul]
8,http://dbpedia.org/resource/%C3%89tranges_Libe...,Étranges Libellules,France,1994-01-01,Lyon,,,Video game industry,,NaN
9,http://dbpedia.org/resource/%C3%96ssur,Össur,Iceland,1971-01-01,Reykjavík,"358,500,000.00","607,100,000.00",Health care,,NaN


In [141]:
forbes = load_csv(INPUT_DIR / "schemamatching" / "original_data" / "forbes.csv", delimiter="\t")
forbes.attrs["dataset_name"] = "forbes"
forbes.head()

,Company,Sector,Industry,Continent,Country,Market Value,Sales,Profits,Assets,Rank,Identifier
0,ICBC,Financials,Major Banks,Asia,CHN,215.6,148.7,42.7,3124.9,1,http://www.forbes.com/companies/icbc/
1,China Construction Bank,Financials,Regional Banks,Asia,CHN,174.4,121.3,34.2,2449.5,2,http://www.forbes.com/companies/china-construc...
2,Agricultural Bank of China,Financials,Regional Banks,Asia,CHN,141.1,136.4,27.0,2405.4,3,http://www.forbes.com/companies/agricultural-b...
3,JPMorgan Chase,Financials,Major Banks,North America,USA,229.7,105.7,17.3,2435.3,4,http://www.forbes.com/companies/jpmorgan-chase/
4,Berkshire Hathaway,Financials,Investment Services,North America,USA,309.1,178.8,19.5,493.4,5,http://www.forbes.com/companies/berkshire-hath...


In [142]:
fullcontact = load_xml(INPUT_DIR / "schemamatching" / "original_data" / "fullcontact.xml")
fullcontact.attrs["dataset_name"] = "fullcontact"
fullcontact.head()

,id,name,onlinesince,country,locality,keypersons_person_name,keypersons_person_title,founded,address
0,fullcontact_1,BBMG,2003-09-22,United States,Brooklyn,Raphael Bemporad,Co-Founder and Principal,NaN,NaN
1,fullcontact_2,CIT Group Inc (DEL),1994-08-19,Canada,Toronto,"[Christine L. Reilly, Jeff Knittel, John A Tha...","[President, CIT Small Business Lending, Presid...",1908,NaN
2,fullcontact_3,City & National Employment,1998-11-23,United States,Waterloo,NaN,NaN,1957,221 E 4th St
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,null,Ireland,Swords,"[Michael W. Lamach, Paul A. Camuti, Steve Shaw...","[President and CEO, SVP & CTO, Sr. VP and CFO,...",1871,NaN
4,fullcontact_5,Evonik,null,Germany,Essen,"[Klaus Engel, Wolfgang Colberg, Eric Jegers, G...","[CEO, CFO, SVP, Senior Vice President Innovati...",2007,Evonik Industries AG Rellinghauser Straße 1-11


In [143]:
# Create founders column based on keypersons_person_name and keypersons_person_title
# Ensure name/title columns are always list-like or NaN
fullcontact["keypersons_person_name"] = fullcontact["keypersons_person_name"].apply(
    lambda x: [x] if isinstance(x, str) else x
)
fullcontact["keypersons_person_title"] = fullcontact["keypersons_person_title"].apply(
    lambda x: [x] if isinstance(x, str) else x
)

def extract_founders(row):
    names = row["keypersons_person_name"]
    titles = row["keypersons_person_title"]
    if not isinstance(names, list) or not isinstance(titles, list):
        return np.nan

    founders = [
        name
        for name, title in zip(names, titles)
        if isinstance(title, str) and "founder" in title.lower()
    ]
    return founders if founders else np.nan

fullcontact["founders"] = fullcontact.apply(extract_founders, axis=1)
fullcontact.head()


,id,name,onlinesince,country,locality,keypersons_person_name,keypersons_person_title,founded,address,founders
0,fullcontact_1,BBMG,2003-09-22,United States,Brooklyn,[Raphael Bemporad],[Co-Founder and Principal],NaN,NaN,[Raphael Bemporad]
1,fullcontact_2,CIT Group Inc (DEL),1994-08-19,Canada,Toronto,"[Christine L. Reilly, Jeff Knittel, John A Tha...","[President, CIT Small Business Lending, Presid...",1908,NaN,NaN
2,fullcontact_3,City & National Employment,1998-11-23,United States,Waterloo,NaN,NaN,1957,221 E 4th St,NaN
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,null,Ireland,Swords,"[Michael W. Lamach, Paul A. Camuti, Steve Shaw...","[President and CEO, SVP & CTO, Sr. VP and CFO,...",1871,NaN,NaN
4,fullcontact_5,Evonik,null,Germany,Essen,"[Klaus Engel, Wolfgang Colberg, Eric Jegers, G...","[CEO, CFO, SVP, Senior Vice President Innovati...",2007,Evonik Industries AG Rellinghauser Straße 1-11,NaN


## Step 3: LLM-Based Schema Matching

In [144]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

[INFO ] PyDI.schemamatching.llm_based - Initialized LLMBasedSchemaMatcher with 40 sample rows
[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: dbpedia -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T13:03:13.502703Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 8923.99001121521, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3869, "output_tokens": 468, "total_tokens": 4337, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 384}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map i

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,identifier,target_schema,id,0.95,llm_based_matching
1,dbpedia,name,target_schema,name,0.95,llm_based_matching
2,dbpedia,countryName,target_schema,country,0.95,llm_based_matching
3,dbpedia,fyear,target_schema,founded,0.95,llm_based_matching
4,dbpedia,cityName,target_schema,city,0.95,llm_based_matching
5,dbpedia,revenue,target_schema,revenue,0.95,llm_based_matching
6,dbpedia,assets,target_schema,assets,0.95,llm_based_matching
7,dbpedia,industryName,target_schema,industry,0.95,llm_based_matching
8,dbpedia,homepage,target_schema,website,0.95,llm_based_matching
9,dbpedia,founders,target_schema,founders,0.95,llm_based_matching


In [145]:
forbes_mapping = matcher.match(forbes, df_target)
forbes_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: forbes -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T13:03:34.415543Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 20875.037908554077, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3417, "output_tokens": 1040, "total_tokens": 4457, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 960}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map it to null. Represent each mapping as a two-item list like [\"Table A column\", \"Table B col

,source_dataset,source_column,target_dataset,target_column,score,notes
0,forbes,Company,target_schema,name,0.95,llm_based_matching
1,forbes,Industry,target_schema,industry,0.95,llm_based_matching
2,forbes,Country,target_schema,country,0.95,llm_based_matching
3,forbes,Sales,target_schema,revenue,0.95,llm_based_matching
4,forbes,Assets,target_schema,assets,0.95,llm_based_matching
5,forbes,Identifier,target_schema,id,0.95,llm_based_matching


In [146]:
fullcontact_mapping = matcher.match(fullcontact, df_target)
fullcontact_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: fullcontact -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T13:03:52.385838Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 17903.83529663086, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 10385, "output_tokens": 724, "total_tokens": 11109, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 640}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map it to null. Represent each mapping as a two-item list like [\"Table A column\", \"Table 

,source_dataset,source_column,target_dataset,target_column,score,notes
0,fullcontact,id,target_schema,id,0.95,llm_based_matching
1,fullcontact,name,target_schema,name,0.95,llm_based_matching
2,fullcontact,country,target_schema,country,0.95,llm_based_matching
3,fullcontact,locality,target_schema,city,0.95,llm_based_matching
4,fullcontact,founded,target_schema,founded,0.95,llm_based_matching
5,fullcontact,founders,target_schema,founders,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [147]:
translator = SchemaTranslator()
# Translate + normalize each dataset with its own mapping

# Adjust forbes financials from billions to absolute numbers
forbes["Sales"] = forbes["Sales"].apply(lambda x: x * 1e9 if pd.notna(x) else x)
forbes["Assets"] = forbes["Assets"].apply(lambda x: x * 1e9 if pd.notna(x) else x)

# Remove , from dbpedia financials
dbpedia["assets"] = dbpedia["assets"].astype(str).str.replace(",", "", regex=False)
dbpedia["revenue"] = dbpedia["revenue"].astype(str).str.replace(",", "", regex=False)
for col in ["assets", "revenue"]:
    dbpedia[col] = pd.to_numeric(dbpedia[col], errors="coerce")
    dbpedia[col] = dbpedia[col].round().astype("Int64")

spec.set_column("country", country_format="name")

forbes_normalized = translator.translate(
    forbes, forbes_mapping,
    normalize=spec, on_failure="keep"
)

dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

fullcontact_normalized = translator.translate(
    fullcontact, fullcontact_mapping,
    normalize=spec, on_failure="keep"
)

[INFO ] root - Translating 6 columns for 'forbes'
[WARNING] PyDI.normalization.transform - Column 'website' not found in DataFrame


[WARNING] PyDI.normalization.transform - Column 'founded' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'city' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'founders' not found in DataFrame
[INFO ] root - Normalization complete: 1988 values transformed, 1 values failed
[INFO ] root - Translating 10 columns for 'dbpedia'
[INFO ] root - Normalization complete: 19980 values transformed, 322 values failed
[INFO ] root - Translating 6 columns for 'fullcontact'
[WARNING] PyDI.normalization.transform - Column 'website' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'industry' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'assets' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'revenue' not found in DataFrame
[INFO ] root - Normalization complete: 2467 values transformed, 84 values failed


In [148]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head(10)

,id,name,website,founded,country,city,industry,assets,revenue,founders
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,,1970-01-01 00:00:00,France,Castelnaudary,Meat,<NA>,<NA>,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,ÿguas de Portugal,,1993-01-01 00:00:00,Portugal,Lisbon,,<NA>,<NA>,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,ÿnima Estudios,,2002-01-01 00:00:00,Mexico,Mexico City,Animation,<NA>,<NA>,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalik Enerji,,1998-01-01 00:00:00,Turkey,Istanbul,,<NA>,<NA>,[Çalik Holding]
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalik Holding,,1997-01-01 00:00:00,Turkey,Istanbul,,8,3,[Ahmet Çalik]
5,http://dbpedia.org/resource/%C3%87ukurova_(con...,Çukurova (construction firm),,1975-01-01 00:00:00,Turkey,Istanbul,,<NA>,<NA>,NaN
6,http://dbpedia.org/resource/%C3%87ukurova_Holding,Çukurova Holding,,1923-01-01 00:00:00,Turkey,Istanbul,Communication,<NA>,<NA>,NaN
7,http://dbpedia.org/resource/%C3%89lectricit%C3...,Électricité de France,,1946-01-01 00:00:00,France,Paris,Electric utility,240560000000,65170000000,[Marcel Paul]
8,http://dbpedia.org/resource/%C3%89tranges_Libe...,Étranges Libellules,,1994-01-01 00:00:00,France,Lyon,Video game industry,<NA>,<NA>,NaN
9,http://dbpedia.org/resource/%C3%96ssur,Össur,,1971-01-01 00:00:00,Iceland,Reykjavík,Health care,607100000,358500000,NaN


In [149]:
# Inspect normalized forbes dataset (target columns only)
forbes_cols = [c for c in target_columns if c in forbes_normalized.columns]
forbes_normalized[forbes_cols].head(10)

,id,name,country,industry,assets,revenue
0,http://www.forbes.com/companies/icbc/,ICBC,China,Major Banks,3124900000000,148700000000
1,http://www.forbes.com/companies/china-construc...,China Construction Bank,China,Regional Banks,2449500000000,121300000000
2,http://www.forbes.com/companies/agricultural-b...,Agricultural Bank of China,China,Regional Banks,2405400000000,136400000000
3,http://www.forbes.com/companies/jpmorgan-chase/,JPMorgan Chase,United States,Major Banks,2435300000000,105700000000
4,http://www.forbes.com/companies/berkshire-hath...,Berkshire Hathaway,United States,Investment Services,493400000000,178800000000
5,http://www.forbes.com/companies/exxon-mobil/,Exxon Mobil,United States,Oil & Gas Operations,346800000000,394000000000
6,http://www.forbes.com/companies/general-electric/,General Electric,United States,Conglomerates,656600000000,143300000000
7,http://www.forbes.com/companies/wells-fargo/,Wells Fargo,United States,Major Banks,1543000000000,88700000000
8,http://www.forbes.com/companies/bank-of-china/,Bank of China,China,Major Banks,2291800000000,105100000000
9,http://www.forbes.com/companies/petrochina/,PetroChina,China,Oil & Gas Operations,386900000000,328500000000


In [150]:
# Inspect normalized fullcontact dataset (target columns only)
fullcontact_cols = [c for c in target_columns if c in fullcontact_normalized.columns]
fullcontact_normalized[fullcontact_cols].head(10)

,id,name,founded,country,city,founders
0,fullcontact_1,BBMG,NaN,United States,Brooklyn,[Raphael Bemporad]
1,fullcontact_2,CIT Group Inc (DEL),1908-01-01 00:00:00,Canada,Toronto,NaN
2,fullcontact_3,City & National Employment,1957-01-01 00:00:00,United States,Waterloo,NaN
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,1871-01-01 00:00:00,Ireland,Swords,NaN
4,fullcontact_5,Evonik,2007-01-01 00:00:00,Germany,Essen,NaN
5,fullcontact_6,Exon Mobil,NaN,United States,Houston,NaN
6,fullcontact_7,PPG Industries,1883-01-01 00:00:00,United States,Pittsburgh,[John Pitcairn]
7,fullcontact_8,Walmart,1962-01-01 00:00:00,United States,Bentonville,NaN
8,fullcontact_9,Deutsche Börse AG,2001-01-01 00:00:00,Germany,Eschborn,NaN
9,fullcontact_10,Tabbs,NaN,NaN,NaN,NaN


In [151]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
forbes = forbes_normalized[forbes_cols].copy()
fullcontact = fullcontact_normalized[fullcontact_cols].copy()

## Part 2: Data Profiling

In [152]:
from PyDI.utils import DataProfiler

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

Total records across all datasets: 14,023
dbpedia:
  Rows: 10,092
  Columns: 10
  Total nulls: 26,626
  Null percentage: 26.4%
  Null counts per column:
    assets: 9,432 (93.5%)
    revenue: 8,030 (79.6%)
    founders: 9,164 (90.8%)

forbes:
  Rows: 2,000
  Columns: 6
  Total nulls: 54
  Null percentage: 0.4%
  Null counts per column:
    country: 11 (0.5%)
    industry: 43 (2.1%)

fullcontact:
  Rows: 1,931
  Columns: 6
  Total nulls: 3,586
  Null percentage: 31.0%
  Null counts per column:
    founded: 875 (45.3%)
    country: 508 (26.3%)
    city: 464 (24.0%)
    founders: 1,739 (90.1%)



{'rows': 1931,
 'columns': 6,
 'nulls_total': 3586,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'founded': 875,
  'country': 508,
  'city': 464,
  'founders': 1739},
 'dtypes': {'id': 'object',
  'name': 'object',
  'founded': 'object',
  'country': 'object',
  'city': 'object',
  'founders': 'object'}}

### Attribute Coverage Analysis

In [153]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 10 attributes across 3 datasets


📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,forbes_count,forbes_pct,forbes_coverage,forbes_samples,fullcontact_count,fullcontact_pct,fullcontact_coverage,fullcontact_samples,avg_coverage,max_coverage,datasets_with_attribute
0,assets,660/10092,6.5%,0.065398,"[8, 240560000000, 607100000]",2000/2000,100.0%,1.0000,"[3124900000000, 2449500000000, 2405400000000]",0/0,0%,0.000000,N/A,0.355133,1.00000,2
1,city,10092/10092,100.0%,1.000000,"['Castelnaudary', 'Lisbon', 'Mexico City']",0/0,0%,0.0000,N/A,1467/1931,76.0%,0.759710,"['Brooklyn', 'Toronto', 'Waterloo']",0.586570,1.00000,2
2,country,10092/10092,100.0%,1.000000,"['France', 'Portugal', 'Mexico']",1989/2000,99.5%,0.9945,"['China', 'China', 'China']",1423/1931,73.7%,0.736924,"['United States', 'Canada', 'United States']",0.910475,1.00000,3
3,founded,10092/10092,100.0%,1.000000,"[Timestamp('1970-01-01 00:00:00'), Timestamp('...",0/0,0%,0.0000,N/A,1056/1931,54.7%,0.546867,"[Timestamp('1908-01-01 00:00:00'), Timestamp('...",0.515622,1.00000,2
4,founders,928/10092,9.2%,0.091954,"[['Çalik Holding'], ['Ahmet Çalik'], ['Marcel ...",0/0,0%,0.0000,N/A,192/1931,9.9%,0.099430,"[['Raphael Bemporad'], ['John Pitcairn'], ['Do...",0.063795,0.09943,2
5,id,10092/10092,100.0%,1.000000,['http://dbpedia.org/resource/%C3%80_la_Table_...,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",1931/1931,100.0%,1.000000,"['fullcontact_1', 'fullcontact_2', 'fullcontac...",1.000000,1.00000,3
6,industry,10092/10092,100.0%,1.000000,"['Meat', '', 'Animation']",1957/2000,97.9%,0.9785,"['Major Banks', 'Regional Banks', 'Regional Ba...",0/0,0%,0.000000,N/A,0.659500,1.00000,2
7,name,10092/10092,100.0%,1.000000,"['À la Table de Spanghero', 'ÿguas de Portugal...",2000/2000,100.0%,1.0000,"['ICBC', 'China Construction Bank', 'Agricultu...",1931/1931,100.0%,1.000000,"['BBMG', 'CIT Group Inc (DEL)', 'City & Nation...",1.000000,1.00000,3
8,revenue,2062/10092,20.4%,0.204320,"[3, 65170000000, 358500000]",2000/2000,100.0%,1.0000,"[148700000000, 121300000000, 136400000000]",0/0,0%,0.000000,N/A,0.401440,1.00000,2
9,website,10092/10092,100.0%,1.000000,"['', '', '']",0/0,0%,0.0000,N/A,0/0,0%,0.000000,N/A,0.333333,1.00000,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['assets', 'city', 'country', 'founded', 'founders', 'id', 'industry', 'name', 'revenue']


## Part 3: Entity Matching

### Step 1: Blocking

In [154]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
from PyDI.entitymatching import TokenBlocker

token_blocker_f2d = TokenBlocker(
    forbes, dbpedia,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

token_blocker_f2fc = TokenBlocker(
    forbes, fullcontact,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 11042 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1113 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/debugResultsBlocking_TokenBlocker.csv
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2709 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1122 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/

### Step 2: Evaluate Blocking Against Ground Truth

In [156]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 4 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 7 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 9 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 11 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 12 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 12 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 12 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 12 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 14 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 22 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 27 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 42 true matches
[INFO ] root - Processed 130 batches, 129862 pairs, 101 true matches
[INFO ] root -   Pair Completeness: 0.971
[INFO ] root -   Pair Quality:      0.0

{'pair_completeness': 0.9711538461538461,
 'pair_quality': 0.0007777486870678104,
 'reduction_ratio': 0.993566091954023,
 'total_candidates': 129862,
 'total_possible_pairs': 20184000,
 'true_positives_found': 101,
 'total_true_pairs': 104,
 'batches_processed': 130,
 'evaluation_timestamp': '2025-12-12T14:04:22.310160',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_detailed_results.csv']}

In [157]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2fc,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 12 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 15 true matches
[INFO ] root - Processed 30 batches, 29500 pairs, 218 true matches
[INFO ] root -   Pair Completeness: 0.944
[INFO ] root -   Pair Quality:      0.007
[INFO ] root -   Reduction Ratio:   0.992361
[INFO ] root -   True Matches Found: 218/231
[INFO ] root -   Batches Processed:  30
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9437229437229437,
 'pair_quality': 0.007389830508474577,
 'reduction_ratio': 0.9923614707405489,
 'total_candidates': 29500,
 'total_possible_pairs': 3862000,
 'true_positives_found': 218,
 'total_true_pairs': 231,
 'batches_processed': 30,
 'evaluation_timestamp': '2025-12-12T14:04:33.016811',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [158]:
from PyDI.entitymatching import StringComparator
import re

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators_f2d = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='name', 
        similarity_function='levenshtein',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

comparators_f2fc = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [159]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes,
    df_right=dbpedia, 
    candidates=token_blocker_f2d,
    comparators=comparators_f2d,
    weights=[1.0, 1.0, 1.0, 0.3],
    threshold=0.2,
    id_column='id'
)

correspondences_f2fc = matcher.match(
    df_left=forbes,
    df_right=fullcontact, 
    candidates=token_blocker_f2fc,
    comparators=comparators_f2fc,
    weights=[1.0, 0.5],
    threshold=0.1,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 10092 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 10092 elements after 0:00:0.161; 129862 blocked pairs (reduction ratio: 0.993566091954023)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:108.058; found 59470 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 1931 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 1931 elements after 0:00:0.055; 29500 blocked pairs (reduction ratio: 0.9923614707405489)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:12.383; found 27076 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [160]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  101
[INFO ] root -   True Negatives:  60
[INFO ] root -   False Positives: 55
[INFO ] root -   False Negatives: 3
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.735
[INFO ] root -   Precision: 0.647
[INFO ] root -   Recall:    0.971
[INFO ] root -   F1-Score:  0.777


{'precision': 0.6474358974358975,
 'recall': 0.9711538461538461,
 'f1': 0.7769230769230769,
 'accuracy': 0.7351598173515982,
 'true_positives': 101,
 'false_positives': 55,
 'false_negatives': 3,
 'true_negatives': 60,
 'threshold_used': 0.0,
 'total_correspondences': 59470,
 'filtered_correspondences': 59470,
 'evaluation_timestamp': '2025-12-12T14:06:43.587130',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

In [161]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 185 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	146	|	78.92%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	18	|	9.73%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	8	|	4.32%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	3	|	1.62%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	2	|	1.08%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	4	|	2.16%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		22	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		5354	|	1	|	0.54%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,146,78.918919
1,3,18,9.729730
2,4,8,4.324324
3,5,3,1.621622
4,6,2,1.081081
5,7,4,2.162162
6,8,1,0.540541
7,11,1,0.540541
8,22,1,0.540541
9,5354,1,0.540541


In [162]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 185 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [163]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# use Greedy One-To-One Matching to refine results to 1:1 matches
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Filtered correspondences: 59470 -> 59470 (threshold=0.0)
[INFO ] root - Greedy matching: 59470 -> 1406 correspondences (2812 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 59470 -> 1406 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 5828 -> 2812 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  93
[INFO ] root -   True Negatives:  107
[INFO ] root -   False Positives: 8
[INFO ] root -   False Negatives: 11
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.913
[INFO ] root -   Precision: 0.921
[INFO ] root -   Recall:    0.894
[INFO ] root -   F1-Score:  0.907


{'precision': 0.9207920792079208,
 'recall': 0.8942307692307693,
 'f1': 0.9073170731707317,
 'accuracy': 0.91324200913242,
 'true_positives': 93,
 'false_positives': 8,
 'false_negatives': 11,
 'true_negatives': 107,
 'threshold_used': 0.0,
 'total_correspondences': 1406,
 'filtered_correspondences': 1406,
 'evaluation_timestamp': '2025-12-12T14:07:07.651916',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 1406 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1406	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,1406,100.0


In [164]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2fc = clusterer.cluster(correspondences_f2fc)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  217
[INFO ] root -   True Negatives:  406
[INFO ] root -   False Positives: 92
[INFO ] root -   False Negatives: 14
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.855
[INFO ] root -   Precision: 0.702
[INFO ] root -   Recall:    0.939
[INFO ] root -   F1-Score:  0.804
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 284 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	265	|	93.31%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	12	|	4.23%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	3	|	1.06%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 

Now, we can directly use the trained model with PyDIs MLBasedMatcher

## Part 4: Data Fusion

In [165]:
forbes["forbes_id"] = forbes["id"]

# Assign trust scores to datasets
forbes.attrs["trust_score"] = 1
dbpedia.attrs["trust_score"] = 3
fullcontact.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 2,498


## Step 1: Define Fusion Strategy

In [166]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('founders', union)
strategy.add_attribute_fuser('founded', prefer_higher_trust)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'assets' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'revenue' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founders' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founded' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'country' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'city' using rule 'shortest_string'


## Step 2: Run Fusion

In [167]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes, dbpedia, fullcontact],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'company_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 4236 of 4236 unique IDs
[INFO ] PyDI.fusion.engine - Created 11523 record groups from 2498 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 11523 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	978	|	8.49%
[INFO ] PyDI.fusion.engine - 		3	|	760	|	6.60%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     assets: 0.85
[INFO ] PyDI.fusion.engine -     city:

Fused rows: 1,738


,_id,_fusion_sources,_fusion_source_datasets,forbes_id,founders,country,assets,founded,city,industry,name,revenue,name_first_token,id,website,_fusion_confidence,_fusion_metadata
0,http://www.forbes.com/companies/brookfield-ass...,[http://www.forbes.com/companies/brookfield-as...,"[forbes, dbpedia, fullcontact]",http://www.forbes.com/companies/brookfield-ass...,[William Mackenzie (railway entrepreneur)],Canada,183,1899-01-01 00:00:00,Toronto,Real Estate,Brookfield Asset Management,19,brookfield,http://www.forbes.com/companies/brookfield-ass...,,0.810227,"{'forbes_id_rule': 'first_non_null', 'forbes_i..."
1,http://www.forbes.com/companies/merck/,"[http://www.forbes.com/companies/merck/, fullc...","[forbes, fullcontact]",http://www.forbes.com/companies/merck/,None,Germany,28700000000,1668,Darmstadt,Pharmaceuticals,Merck,14700000000,merck,http://www.forbes.com/companies/merck/,NaN,0.700000,"{'forbes_id_rule': 'first_non_null', 'forbes_i..."
2,http://dbpedia.org/resource/Buw_Holding,"[http://dbpedia.org/resource/Buw_Holding, http...","[dbpedia, forbes]",http://www.forbes.com/companies/otsuka-holding/,None,Germany,17900000000,1993-01-01 00:00:00,Lower SaxonyOsnabrück,Business process outsourcing,Buw Holding,14400000000,buw,http://dbpedia.org/resource/Buw_Holding,,0.636364,"{'founders_rule': 'union', 'founders_sources':..."
3,http://dbpedia.org/resource/China_Communicatio...,[http://dbpedia.org/resource/China_Communicati...,"[dbpedia, fullcontact, forbes]",http://www.forbes.com/companies/china-communic...,None,China,9200000000,2006-01-01 00:00:00,Emery,Telecommunication,China Communications Services Corporation,13500000000,china,http://dbpedia.org/resource/China_Communicatio...,,0.632576,"{'founders_rule': 'union', 'founders_sources':..."
4,http://www.forbes.com/companies/qatar-national...,[http://www.forbes.com/companies/qatar-nationa...,"[forbes, dbpedia]",http://www.forbes.com/companies/qatar-national...,None,Qatar,107800000000,1974-01-01 00:00:00,Doha,Regional Banks,Qatar National Bank,51600000000,qatar,http://www.forbes.com/companies/qatar-national...,,0.681818,"{'forbes_id_rule': 'first_non_null', 'forbes_i..."


## Step 3: Evaluate Data Fusion

In [168]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("founders", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'revenue' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founders'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founded'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'city'


In [169]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
# rename keypeople_name to founders for evaluation
fusion_test_set['founders'] = fusion_test_set['keypeople_name'].apply(lambda x: [x] if isinstance(x, str) else x)

# convert scientific number notation into integers
def convert_scientific_notation(value):
    try:
        if isinstance(value, str) and ('e' in value or 'E' in value):
            return int(float(value))
        return value
    except:
        return value

fusion_test_set['assets'] = fusion_test_set['assets'].apply(convert_scientific_notation)
fusion_test_set['revenue'] = fusion_test_set['revenue'].apply(convert_scientific_notation) 

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.838 overall accuracy (98/117)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 19 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	revenue                          |       5 |     26.32%%
[INFO ] PyDI.fusion.evaluation - 	founders                         |       4 |     21.05%%
[INFO ] PyDI.fusion.evaluation - 	city                             |       4 |     21.05%%
[INFO ] PyDI.fusion.evaluation - 	country                          |       2 |     10.53%%
[INFO ] PyDI.fusion.eval

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.838
  macro_accuracy: 0.817
  num_evaluated_records: 18
  num_evaluated_attributes: 7
  total_evaluations: 117
  total_correct: 98
  founders_accuracy: 0.556
  founders_count: 9
  assets_accuracy: 0.944
  assets_count: 18
  country_accuracy: 0.889
  country_count: 18
  founded_accuracy: 0.889
  founded_count: 18
  city_accuracy: 0.778
  city_count: 18
  name_accuracy: 0.944
  name_count: 18
  revenue_accuracy: 0.722
  revenue_count: 18

Overall Accuracy: 83.8%
